# Editing primitives

`insert()`, `replace()`, and `delete()` edit a `SequenceGraph` in place and return the graph-space `Locus` of the change, recorded as its own operation.

In [1]:
from pathlib import Path
import tempfile

import gen

root = Path(tempfile.mkdtemp(prefix="gen-editing-"))
fasta = root / "reference.fa"
fasta.write_text(">chr1\nAACCGGTTAACCGGTT\n")
repository = gen.Repository(str(root / "repository"))
graph = repository.import_fasta(str(fasta), sample="reference")[0]

In [2]:
hit = graph.search("CCGG", sequence_kind="exact")[0]
inserted = graph.replace(hit, "TT")

assert graph.search("AATTTT", sequence_kind="exact")
inserted

Locus([0f895834[0:2][:]], strand='+')

# Stacking bubbles

`insert()`, `replace()`, and `delete()` all take a `stack` option (default `False`). A plain edit reuses the `chromosome_index` already at the target, so a second edit there supersedes the first rather than coexisting with it — this is also why editing the same locus twice never builds a bubble. `stack=True` writes the new edit under a `chromosome_index` that's never entered into that competition, so the original bases stay just as reachable as the new ones: a real bubble with two live options, not one option replacing the other.

A stacked edit never becomes the reference: the sequence graph's current Path is left exactly as it was, even when the target lies on it, since nothing has been decided about which option is canonical.

In [3]:
bubble_fasta = root / "bubble_reference.fa"
bubble_fasta.write_text(">chr1\nAACCGGTTAACCGGTT\n")
bubble_repository = gen.Repository(str(root / "bubble_repository"))
bubble_graph = bubble_repository.import_fasta(str(bubble_fasta), sample="reference")[0]

hit = bubble_graph.search("CCGG", sequence_kind="exact")[0]
bubble_graph.replace(hit, "TT", stack=True)

# Both routes through this locus are live at once — nothing was superseded.
assert bubble_graph.search("AACCGGTTAACCGGTT", sequence_kind="exact")  # original
assert bubble_graph.search("AATTTTAACCGGTT", sequence_kind="exact")  # stacked option

# The Path (and export_fasta) still reads the original, unedited sequence --
# neither option has been chosen as the reference yet.
export_path = root / "bubble_export.fa"
bubble_graph.export_fasta(str(export_path))
export_path.read_text()

Exported to file /var/folders/f8/8zf8xczs0pxf9_vfnlmqlx9h0000gn/T/gen-editing-yczquvj5/bubble_export.fa


'>chr1\nAACCGGTTAACCGGTT\n'